# HDFS Log Embedding Export

Embed raw log lines, mean-pool per BlockId, save as CSV.
One row per block: `block_id, dim_0, dim_1, ..., dim_n`

In [ ]:
# !pip install sentence-transformers


In [ ]:
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

DATA_DIR     = Path("../../HDFS_v1")
RAW_LOG_FILE = DATA_DIR / "HDFS.log"
OUTPUT_DIR   = Path("../data/embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS_TO_EXPORT = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "BAAI/bge-small-en-v1.5",
    "intfloat/e5-base-v2",
]

BATCH_SIZE = 256
BLOCKID_RE = re.compile(r'(blk_-?\d+)')
print("Config OK")


In [ ]:
# ── Load raw logs ─────────────────────────────────────────────────────────
print("Loading log lines...")
rows = []
with open(RAW_LOG_FILE) as f:
    for line in f:
        line = line.strip()
        m = BLOCKID_RE.search(line)
        if m:
            rows.append({"block_id": m.group(1), "text": line})

log_df = pd.DataFrame(rows)
print(f"Lines with BlockId : {len(log_df):,}")
print(f"Unique blocks      : {log_df['block_id'].nunique():,}")


In [ ]:
# ── Embed & mean-pool per block ───────────────────────────────────────────
def embed_and_export(log_df: pd.DataFrame, model_name: str, batch_size: int = BATCH_SIZE):
    safe_name = model_name.replace("/", "_")
    out_path  = OUTPUT_DIR / f"embeddings_{safe_name}.csv"

    if out_path.exists():
        print(f"  Skipping {model_name} — {out_path} already exists")
        return

    print(f"\nLoading {model_name}...")
    model = SentenceTransformer(model_name)

    groups    = log_df.groupby("block_id")["text"].apply(list)
    block_ids = list(groups.index)

    all_lines, line_to_blk = [], []
    for blk_idx, lines in enumerate(groups):
        all_lines.extend(lines)
        line_to_blk.extend([blk_idx] * len(lines))
    line_to_blk = np.array(line_to_blk)

    print(f"  Encoding {len(all_lines):,} lines...")
    t0        = time.time()
    line_embs = model.encode(
        all_lines,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    d          = line_embs.shape[1]
    block_embs = np.zeros((len(block_ids), d), dtype=np.float32)
    counts     = np.zeros(len(block_ids), dtype=np.int32)
    np.add.at(block_embs, line_to_blk, line_embs)
    np.add.at(counts,     line_to_blk, 1)
    block_embs /= counts[:, None]

    print(f"  Encoding done ({time.time()-t0:.0f}s) — shape {block_embs.shape}")

    # Save: block_id + one column per embedding dimension
    dim_cols = [f"dim_{i}" for i in range(d)]
    out_df   = pd.DataFrame(block_embs, columns=dim_cols)
    out_df.insert(0, "block_id", block_ids)
    out_df.to_csv(out_path, index=False)
    print(f"  Saved → {out_path}  ({out_df.shape[0]:,} rows x {d} dims)")

    del model, line_embs, block_embs
    import gc; gc.collect()


for model_name in MODELS_TO_EXPORT:
    embed_and_export(log_df, model_name)

print("\nAll exports done.")
